<a href="https://colab.research.google.com/github/felicity8899/mlops/blob/main/homework2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [58]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge

In [35]:
df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv')

In [36]:
columns_to_keep = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year',
    'fuel_efficiency_mpg',
]

df = df[columns_to_keep]

In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   engine_displacement  10000 non-null  int64  
 1   horsepower           9123 non-null   float64
 2   vehicle_weight       10000 non-null  int64  
 3   model_year           10000 non-null  int64  
 4   fuel_efficiency_mpg  10000 non-null  float64
dtypes: float64(2), int64(3)
memory usage: 390.8 KB


In [38]:
df.horsepower.describe()

,horsepower
count,9123.000000
mean,254.446016
std,22.844613
min,171.000000
25%,239.000000
50%,254.000000
75%,270.000000
max,334.000000


In [52]:
def prepare_split_data(df):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(42)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]
    df_test = df.iloc[idx[n_train + n_val:]]

    return df_train, df_val, df_test

def rmse(y, y_pred):
    error = y_pred - y
    mse = (error ** 2).mean()
    return np.sqrt(mse)

def prepare_X(df, fill_value):
    df_num = df.copy()
    df_num['horsepower'] = df_num['horsepower'].fillna(fill_value)
    return df_num

**fill with 0 **

In [49]:
df_0 = prepare_X(df, 0)
df_0.describe()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,2268.807000,232.131100,4286.754000,1999.549500,29.997700
std,183.447355,75.210419,266.212491,14.207925,2.930245
min,1590.000000,0.000000,3320.000000,1975.000000,19.800000
25%,2150.000000,234.000000,4110.000000,1987.000000,28.000000
50%,2270.000000,252.000000,4280.000000,2000.000000,30.000000
75%,2390.000000,268.000000,4470.000000,2012.000000,31.900000
max,3110.000000,334.000000,5460.000000,2024.000000,41.200000


In [41]:
df_train, df_val, df_test = prepare_split_data(df_0)

In [53]:
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values

X_train_0 = df_train.drop('fuel_efficiency_mpg', axis=1).values
X_val_0 = df_val.drop('fuel_efficiency_mpg', axis=1).values

lr_0 = LinearRegression()
lr_0.fit(X_train_0, y_train)

y_pred_0 = lr_0.predict(X_val_0)

score_0 = round(rmse(y_val, y_pred_0), 5)

Mean

In [55]:
mean_val = df_train['horsepower'].mean()

df_mean = prepare_X(df, mean_val)
df_mean.describe()
df_train, df_val, df_test = prepare_split_data(df_mean)

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values

X_train_mean = df_train.drop('fuel_efficiency_mpg', axis=1).values
X_val_mean = df_val.drop('fuel_efficiency_mpg', axis=1).values

lr_mean = LinearRegression()
lr_mean.fit(X_train_mean, y_train)

y_pred_mean = lr_mean.predict(X_val_mean)

score_mean = round(rmse(y_val, y_pred_mean), 5)

print(f"RMSE with 0: {score_0}")
print(f"RMSE with mean: {score_mean}")

RMSE with 0: 2.20529
RMSE with mean: 2.20122


In [71]:
r_values = [0, 0.01, 0.1, 1, 5, 10, 100]

results = []

for r in r_values:
    model = Ridge(alpha=r, solver='svd')
    model.fit(X_train_0, y_train)

    y_pred = model.predict(X_val_0)
    score = np.sqrt(rmse(y_val, y_pred))

    results.append((r, score))
    print(f"r: {r:<6} -> RMSE: {score}")
best_r, best_score = min(results, key=lambda x: x[1])

print("-" * 25)
print(f"Best r: {best_r} with RMSE: {best_score}")

r: 0      -> RMSE: 1.485022961018302
r: 0.01   -> RMSE: 1.4850229610336534
r: 0.1    -> RMSE: 1.4850229611718122
r: 1      -> RMSE: 1.485022962553508
r: 5      -> RMSE: 1.485022968696666
r: 10     -> RMSE: 1.4850229763808616
r: 100    -> RMSE: 1.485023115693414
-------------------------
Best r: 0 with RMSE: 1.485022961018302


In [73]:
features = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']
target = 'fuel_efficiency_mpg'

seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
rmse_scores = []

n = len(df)
n_val = int(0.2 * n)
n_test = int(0.2 * n)
n_train = n - n_val - n_test

for s in seeds:
    np.random.seed(s)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_shuffled = df.iloc[idx]

    df_train = df_shuffled.iloc[:n_train].copy()
    df_val = df_shuffled.iloc[n_train:n_train + n_val].copy()
    df_test = df_shuffled.iloc[n_train + n_val:].copy()

    y_train = df_train[target].values
    y_val = df_val[target].values

    X_train = df_train[features].fillna({'horsepower': 0}).values
    X_val = df_val[features].fillna({'horsepower': 0}).values

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    score = np.sqrt(mean_squared_error(y_val, y_pred))
    rmse_scores.append(score)

std_score = round(float(np.std(rmse_scores)), 3)

print("Scores:", [round(s, 4) for s in rmse_scores])
print(f"std: {std_score}")

Scores: [np.float64(2.2392), np.float64(2.2021), np.float64(2.1634), np.float64(2.2044), np.float64(2.2019), np.float64(2.2458), np.float64(2.2697), np.float64(2.1908), np.float64(2.2166), np.float64(2.207)]
std: 0.029


In [74]:


features = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']
target = 'fuel_efficiency_mpg'

n = len(df)
n_val = int(0.2 * n)
n_test = int(0.2 * n)
n_train = n - n_val - n_test

np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)

df_shuffled = df.iloc[idx]

df_train = df_shuffled.iloc[:n_train].copy()
df_val = df_shuffled.iloc[n_train:n_train + n_val].copy()
df_test = df_shuffled.iloc[n_train + n_val:].copy()

df_full_train = pd.concat([df_train, df_val]).reset_index(drop=True)

X_full_train = df_full_train[features].fillna({'horsepower': 0}).values
y_full_train = df_full_train[target].values

X_test = df_test[features].fillna({'horsepower': 0}).values
y_test = df_test[target].values

def train_linear_regression_reg(X, y, r=0.001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    reg = r * np.eye(XTX.shape[0])
    XTX = XTX + reg

    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)

    return w[0], w[1:]

w_0, w = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)

y_pred = w_0 + X_test.dot(w)
rmse_test = np.sqrt(((y_test - y_pred) ** 2).mean())

print(f"Test RMSE: {round(rmse_test, 3)}")

Test RMSE: 2.236
